<a href="https://colab.research.google.com/github/ajzal4you/Master-Project-/blob/main/OCR_MODULE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
ROOT = "/content/drive/MyDrive/MSc_Project/ocr_module"
import os
DATA_DIR   = os.path.join(ROOT, "data")
MODELS_DIR = os.path.join(ROOT, "models")
LOGS_DIR   = os.path.join(ROOT, "logs")
SAMPLES_DIR    = os.path.join(ROOT, "data", "samples")
TEXT_OUT_DIR   = os.path.join(ROOT, "outputs", "text")
STRUCT_OUT_DIR = os.path.join(ROOT, "outputs", "structured")
print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("LOGS_DIR:", LOGS_DIR)


ROOT: /content/drive/MyDrive/MSc_Project/ocr_module
DATA_DIR: /content/drive/MyDrive/MSc_Project/ocr_module/data
MODELS_DIR: /content/drive/MyDrive/MSc_Project/ocr_module/models
LOGS_DIR: /content/drive/MyDrive/MSc_Project/ocr_module/logs


In [ ]:
import os

base_path = "/content/ocr_module"

folders = [
    "data/raw_images",
    "data/processed",
    "models/trocr_en",
    "models/trocr_ml",     # For Malayalam
    "models/trocr_ar",     # For Arabic
    "models/trocr_hi",     # For Hindi
    "scripts",
    "results/extracted_text",
    "config"
]

for folder in folders:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

print("OCR module folder structure created successfully.")

OCR module folder structure created successfully.


In [ ]:
!pip install transformers
!pip install torch torchvision torchaudio
!pip install datasets
!pip install sentencepiece
!pip install pillow
!pip install transformers datasets pillow

In [ ]:
import os
import torch
import requests
import pandas as pd

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from transformers import VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import default_data_collator
from PIL import Image
from datasets import load_dataset, Dataset
from datasets import Dataset
from transformers import TrOCRProcessor

In [ ]:
# Load the labels file
df = pd.read_csv("/content/labels_clean.csv")
dataset = Dataset.from_pandas(df)\


#unzip the file
!unzip /content/sample_images.zip -d /content/


# Show preview
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/labels_clean.csv'

In [ ]:
!ls /content/sample_*.png

In [ ]:
!ls /content/images/ | grep sample

In [ ]:
import pandas as pd

# File names and labels
image_files = [f"sample_{i}.png" for i in range(1, 6)]
texts = ["Hello", "My name is Ajzal", "This is a test", "Artificial Intelligence", "I am learning"]

# Create new DataFrame
df = pd.DataFrame({"image_path": image_files, "text": texts})

# Save CSV to /content/
df.to_csv("/content/labels_fixed.csv", index=False)

print(" New labels_fixed.csv saved successfully.")


In [ ]:
# Load fixed CSV
df = pd.read_csv("/content/labels_fixed.csv")
dataset = Dataset.from_pandas(df)

In [ ]:
# Load TrOCR processor
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")

In [ ]:
# Preprocessing function
def preprocess(example):
    image = Image.open(f"/content/{example['image_path']}").convert("RGB")
    # Ensure text is passed to the processor
    encoding = processor(images=image, text=str(example["text"]), padding="max_length", truncation=True, return_tensors="pt")
    # The processor should return a dictionary with 'pixel_values' and 'input_ids'
    # We are using the 'input_ids' as the labels for the model
    if "input_ids" in encoding:
        encoding["labels"] = encoding["input_ids"].squeeze()
    return encoding

In [ ]:
# Map preprocessing
processed_dataset = dataset.map(preprocess)
print(" Dataset is preprocessed and ready for fine-tuning.")

In [ ]:
# Load base TrOCR model (pretrained)
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/trocr_finetuned",     # where to save the model
    per_device_train_batch_size=2,
    num_train_epochs=5,
    logging_steps=5,
    save_steps=100,
    predict_with_generate=True,
    fp16=torch.cuda.is_available()
)
print("Training arguments set.")

In [ ]:
# Define Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset,
    data_collator=default_data_collator,
    tokenizer=processor.feature_extractor,
)

trainer.train()

print(" Trainer ready.")

In [ ]:
model.save_pretrained("/content/trocr_finetuned")
processor.save_pretrained("/content/trocr_finetuned")
print(" Model and processor saved to /content/trocr_finetuned")

In [ ]:
# Load model & processor
model = VisionEncoderDecoderModel.from_pretrained("/content/trocr_finetuned")
processor = TrOCRProcessor.from_pretrained("/content/trocr_finetuned")

# Load an image
image = Image.open("/content/sample_1.png").convert("RGB")

# Run OCR
pixel_values = processor(images=image, return_tensors="pt").pixel_values
generated_ids = model.generate(pixel_values)
predicted_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

print(" Predicted Text:", predicted_text)

In [ ]:
!zip -r /content/trocr_finetuned.zip /content/trocr_finetuned

In [ ]:
print(" New labels_fixed.csv saved successfully.")
print(" Dataset is preprocessed and ready for fine-tuning.")
print("Training arguments set.")
print(" Trainer ready.")
print(" Model saved")
print(" Model and processor saved to /content/trocr_finetuned")
print(" Predicted Text:", predicted_text)